# Sliced Wasserstein Projections

Sliced Wasserstein replaces one high-dimensional OT problem by many one-dimensional projected OT problems.  For a direction $\theta$, one compares
$$
    (P_\theta)_\sharp\alpha,\qquad (P_\theta)_\sharp\beta,
    \qquad P_\theta(x)=\langle x,\theta\rangle.
$$
This notebook uses two simple shape densities and displays five projected histograms for each density.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks-figures":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks-figures"))

from figure_style import (
    BLUE,
    RED,
    VIOLET,
    ORANGE,
    GRAY,
    LIGHT_GRAY,
    DIRAC_MARKER_SIZE,
    box_axes,
    draw_point_clouds,
    draw_transport_segments,
    figure_dir,
    interp_color,
    padded_limits,
    remove_axes,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()

from PIL import Image

NAME = "sliced-wasserstein-projections"
out = figure_dir(NAME)
rng = np.random.default_rng(12)
assets = ROOT / "notebooks-figures" / "assets"

The density panels show samples from the shapes.  The middle panels stack the one-dimensional histograms along five fixed projection directions; no optimization is involved.

In [2]:
def sample_from_image(path, n, *, invert=False):
    img = Image.open(path).convert("RGBA")
    arr = np.asarray(img)
    gray = np.asarray(img.convert("L"), dtype=float)
    alpha = arr[..., 3]
    # Some PNG assets have an almost opaque white background, so only trust
    # alpha when the transparency contrast is large.  Otherwise use the dark
    # silhouette as foreground.
    if alpha.max() - alpha.min() > 100 and alpha.min() < 80:
        mask = alpha > 80
    else:
        mask = gray < 128 if not invert else gray > 128
    yy, xx = np.nonzero(mask)
    if len(xx) < n:
        raise RuntimeError(f"not enough mask pixels in {path}")
    idx = rng.choice(len(xx), size=n, replace=False)
    pts = np.column_stack([xx[idx], -yy[idx]]).astype(float)
    pts -= pts.mean(axis=0, keepdims=True)
    pts /= np.max(np.linalg.norm(pts, axis=1))
    return pts

n = 850
alpha_pts = sample_from_image(assets / "cat.png", n)
beta_pts = sample_from_image(assets / "heart.jpg", n)
# Normalize both shapes to the same visual scale and separate orientation slightly.
alpha_pts = alpha_pts @ np.array([[0.96, -0.06], [0.06, 0.96]]).T
beta_pts = beta_pts @ np.array([[0.92, 0.10], [-0.10, 0.92]]).T
angles = np.deg2rad([-60, -25, 10, 45, 80])
directions = np.column_stack([np.cos(angles), np.sin(angles)])
dir_colors = [GRAY, ORANGE, VIOLET, "#8c6d31", "#1b7837"]

In [3]:
def draw_density(points, color, filename):
    fig, ax = plt.subplots(figsize=(2.12, 2.12))
    ax.scatter(points[:,0], points[:,1], s=DIRAC_MARKER_SIZE * 0.11, marker="o", color=color, edgecolor="none", linewidth=0, alpha=0.42)
    for theta, c in zip(directions, dir_colors):
        line = np.vstack([-0.92*theta, 0.92*theta])
        ax.plot(line[:,0], line[:,1], color=c, lw=0.78, alpha=0.58, zorder=2)
    ax.set_aspect("equal")
    ax.set_xlim(-1.07, 1.07)
    ax.set_ylim(-1.07, 1.07)
    remove_axes(ax)
    save_pdf(fig, out / filename, pad_inches=0.035)
    plt.close(fig)

def draw_hist_stack(points, color, filename):
    fig, ax = plt.subplots(figsize=(1.60, 2.12))
    bins = np.linspace(-1.05, 1.05, 44)
    centers = 0.5 * (bins[:-1] + bins[1:])
    for k, (theta, c) in enumerate(zip(directions, dir_colors)):
        proj = points @ theta
        h, _ = np.histogram(proj, bins=bins, density=True)
        h = h / max(h.max(), 1e-12)
        base = len(directions) - 1 - k
        ax.fill_between(centers, base, base + 0.58*h, color=color, alpha=0.18, linewidth=0)
        ax.plot(centers, base + 0.58*h, color=color, lw=0.82, alpha=0.92)
        ax.plot([bins[0], bins[-1]], [base, base], color=c, lw=0.72, alpha=0.75)
    ax.set_xlim(bins[0], bins[-1])
    ax.set_ylim(-0.18, len(directions)-0.20)
    remove_axes(ax)
    save_pdf(fig, out / filename, pad_inches=0.035)
    plt.close(fig)

draw_density(alpha_pts, RED, "density-alpha.pdf")
draw_hist_stack(alpha_pts, RED, "hist-alpha.pdf")
draw_hist_stack(beta_pts, BLUE, "hist-beta.pdf")
draw_density(beta_pts, BLUE, "density-beta.pdf")
# Backward-compatible aliases for the old gallery card.
draw_density(np.vstack([alpha_pts[:350], beta_pts[:350]]), VIOLET, "directions.pdf")
draw_hist_stack(np.vstack([alpha_pts[:350], beta_pts[:350]]), VIOLET, "projected-matchings.pdf")